In [7]:

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
import warnings
warnings.filterwarnings("ignore")
import os
import glob
import re
from datetime import datetime
from collections import defaultdict
# Import visualization libraries
import cartopy.crs as ccrs
import cartopy.feature as cfeature


print("Libraries imported successfully!")

Libraries imported successfully!


In [8]:
# Replace the file selection cell with this:

# Data directory
DATA_DIR = r"Y:\unsmooth data rev d\SWOT_L2_LR_SSH_D_D-20260131_185119"

# Date range filter
START_DATE = datetime(2022, 8, 13)   
END_DATE = datetime(2026, 8, 30)      

# Number of files to analyze
NUM_FILES = 40

def parse_swot_filename(filename):
    """
    Parse SWOT filename to extract metadata.
    Example: SWOT_L2_LR_SSH_Unsmoothed_002_077_20230813T192851_20230813T202018_PGC0_01.nc
    """
    basename = os.path.basename(filename)
    
    pattern = r'SWOT_L2_LR_SSH_Unsmoothed_(\d{3})_(\d{3})_(\d{8}T\d{6})_(\d{8}T\d{6})_([A-Z0-9]+)_(\d{2})\.nc'
    match = re.match(pattern, basename)
    
    if match:
        cycle = int(match.group(1))
        pass_number = int(match.group(2))
        start_str = match.group(3)
        end_str = match.group(4)
        baseline = match.group(5)
        product_counter = int(match.group(6))
        
        start_dt = datetime.strptime(start_str, '%Y%m%dT%H%M%S')
        end_dt = datetime.strptime(end_str, '%Y%m%dT%H%M%S')
        
        return {
            'filename': basename,
            'filepath': filename,
            'cycle': cycle,
            'pass_number': pass_number,
            'start_datetime': start_dt,
            'end_datetime': end_dt,
            'baseline': baseline,
            'product_counter': product_counter
        }
    return None

# Get all NC files
all_files = glob.glob(os.path.join(DATA_DIR, "*.nc"))
print(f"Total files found: {len(all_files)}")

# Parse all files
parsed_files = [parse_swot_filename(f) for f in all_files if parse_swot_filename(f)]
print(f"Successfully parsed: {len(parsed_files)} files")

# Filter by date range
filtered_by_date = [
    f for f in parsed_files 
    if START_DATE <= f['start_datetime'] <= END_DATE
]
print(f"Files in date range: {len(filtered_by_date)}")

# Group by unique data granule and keep HIGHEST product counter
granule_groups = defaultdict(list)
for f in filtered_by_date:
    key = (f['cycle'], f['pass_number'], f['start_datetime'].strftime('%Y%m%dT%H%M%S'))
    granule_groups[key].append(f)

# Select highest product counter for each granule
unique_files = []
for key, files in granule_groups.items():
    best_file = max(files, key=lambda x: x['product_counter'])
    unique_files.append(best_file)

# Sort by date
unique_files.sort(key=lambda x: x['start_datetime'])

# ===================================================================
# SHOW ALL AVAILABLE PASSES
# ===================================================================
available_passes = sorted(set(f['pass_number'] for f in unique_files))
pass_counts = {p: sum(1 for f in unique_files if f['pass_number'] == p)
               for p in available_passes}

print("\n" + "=" * 80)
print("ALL AVAILABLE PASSES (SORTED BY PASS NUMBER)")
print("=" * 80)
print(f"{'Pass':<10} {'Files':<10}")
print("-" * 80)

for p in available_passes:
    print(f"{p:<10} {pass_counts[p]:<10}")

print("=" * 80)
print(f"Total passes: {len(available_passes)}")




Total files found: 0
Successfully parsed: 0 files
Files in date range: 0

ALL AVAILABLE PASSES (SORTED BY PASS NUMBER)
Pass       Files     
--------------------------------------------------------------------------------
Total passes: 0


In [9]:
# # ===================================================================
# # VISUALIZE ALL PASSES ON A SINGLE MAP
# # ===================================================================

# def visualize_all_passes(unique_files, lat_bounds=(5.0, 20.0), lon_bounds=(92.0, 100.0)):
#     """
#     Visualize all unique passes on a single map.
#     Uses minimum cycle number and PGC0_01 baseline for each pass.
    
#     Parameters:
#     -----------
#     unique_files : list
#         List of parsed file dictionaries
#     lat_bounds : tuple
#         (lat_min, lat_max) for target region
#     lon_bounds : tuple
#         (lon_min, lon_max) for target region
#     """
    
#     # Get all unique passes
#     all_passes = sorted(set(f['pass_number'] for f in unique_files))
#     print(f"Total unique passes: {len(all_passes)}")
#     print(f"Passes: {all_passes}")
    
#     # Select one file per pass: minimum cycle, prefer PGC0 baseline, product counter 01
#     selected_pass_files = []
    
#     for pass_id in all_passes:
#         # Get all files for this pass
#         pass_files = [f for f in unique_files if f['pass_number'] == pass_id]
        
#         # Prefer PGC0 baseline files
#         pgc0_files = [f for f in pass_files if 'PGC0' in f['baseline']]
        
#         if pgc0_files:
#             # Select minimum cycle from PGC0 files
#             best_file = min(pgc0_files, key=lambda x: (x['cycle'], -x['product_counter']))
#         else:
#             # Fallback: select minimum cycle from all files
#             best_file = min(pass_files, key=lambda x: (x['cycle'], -x['product_counter']))
        
#         selected_pass_files.append(best_file)
#         print(f"  Pass {pass_id:3d}: Cycle {best_file['cycle']:3d}, {best_file['baseline']}")
    
#     print(f"\nLoading {len(selected_pass_files)} files (one per pass)...")
    
#     # Create figure
#     fig = plt.figure(figsize=(20, 12))
    
#     # Main map with all passes
#     ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    
#     # Set extent to show Andaman Sea region with some buffer
#     buffer = 5
#     ax.set_extent([lon_bounds[0] - buffer, lon_bounds[1] + buffer, 
#                    lat_bounds[0] - buffer, lat_bounds[1] + buffer], 
#                   crs=ccrs.PlateCarree())
    
#     # Add map features
#     ax.add_feature(cfeature.LAND, facecolor='lightgray', edgecolor='black', linewidth=0.5)
#     ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3)
#     ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
#     ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
#     ax.add_feature(cfeature.RIVERS, linewidth=0.3, edgecolor='blue', alpha=0.5)
    
#     # Generate distinct colors for each pass
#     cmap = plt.cm.get_cmap('tab20', len(all_passes))
#     colors = [cmap(i) for i in range(len(all_passes))]
    
#     # Track which passes cover the target region
#     passes_in_region = []
#     passes_outside_region = []
    
#     FILL_THRESHOLD = 1e10
    
#     # Load and plot each pass
#     for idx, (file_info, color) in enumerate(zip(selected_pass_files, colors)):
#         pass_id = file_info['pass_number']
        
#         try:
#             # Load swath data
#             ds_left = xr.open_dataset(file_info['filepath'], group='left', engine='netcdf4')
#             ds_right = xr.open_dataset(file_info['filepath'], group='right', engine='netcdf4')
            
#             # Get coordinates
#             lat_left = ds_left['latitude'].values.copy()
#             lon_left = ds_left['longitude'].values.copy()
#             lat_right = ds_right['latitude'].values.copy()
#             lon_right = ds_right['longitude'].values.copy()
            
#             # Clean fill values
#             lat_left = np.where(np.abs(lat_left) > FILL_THRESHOLD, np.nan, lat_left)
#             lon_left = np.where(np.abs(lon_left) > FILL_THRESHOLD, np.nan, lon_left)
#             lat_right = np.where(np.abs(lat_right) > FILL_THRESHOLD, np.nan, lat_right)
#             lon_right = np.where(np.abs(lon_right) > FILL_THRESHOLD, np.nan, lon_right)
            
#             # Get valid points
#             valid_left = np.isfinite(lat_left) & np.isfinite(lon_left)
#             valid_right = np.isfinite(lat_right) & np.isfinite(lon_right)
            
#             # Flatten valid data
#             lat_left_flat = lat_left[valid_left]
#             lon_left_flat = lon_left[valid_left]
#             lat_right_flat = lat_right[valid_right]
#             lon_right_flat = lon_right[valid_right]
            
#             # Combine both swaths
#             all_lats = np.concatenate([lat_left_flat, lat_right_flat])
#             all_lons = np.concatenate([lon_left_flat, lon_right_flat])
            
#             # Check if pass is in target region
#             in_region = ((all_lats >= lat_bounds[0]) & (all_lats <= lat_bounds[1]) & 
#                         (all_lons >= lon_bounds[0]) & (all_lons <= lon_bounds[1]))
#             points_in_region = np.sum(in_region)
            
#             if points_in_region > 0:
#                 passes_in_region.append((pass_id, points_in_region))
#             else:
#                 passes_outside_region.append(pass_id)
            
#             # Subsample for plotting
#             step = 20
            
#             # Plot both swaths with same color for this pass
#             ax.scatter(lon_left_flat[::step], lat_left_flat[::step], 
#                       c=[color], s=1, alpha=0.7, transform=ccrs.PlateCarree())
#             ax.scatter(lon_right_flat[::step], lat_right_flat[::step], 
#                       c=[color], s=1, alpha=0.7, label=f'Pass {pass_id}',
#                       transform=ccrs.PlateCarree())
            
#             ds_left.close()
#             ds_right.close()
            
#         except Exception as e:
#             print(f"  ⚠️ Error loading Pass {pass_id}: {str(e)[:50]}")
    
#     # Draw target region box (Andaman Sea)
#     target_lons = [lon_bounds[0], lon_bounds[1], lon_bounds[1], lon_bounds[0], lon_bounds[0]]
#     target_lats = [lat_bounds[0], lat_bounds[0], lat_bounds[1], lat_bounds[1], lat_bounds[0]]
#     ax.plot(target_lons, target_lats, 'k-', linewidth=3, label='Andaman Sea Region')
#     ax.fill(target_lons, target_lats, alpha=0.1, color='yellow')
    
#     # Add region label
#     ax.text((lon_bounds[0] + lon_bounds[1])/2, (lat_bounds[0] + lat_bounds[1])/2, 
#             'ANDAMAN\nSEA', fontsize=14, fontweight='bold', ha='center', va='center',
#             color='darkred', transform=ccrs.PlateCarree(),
#             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
#     # Gridlines
#     gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
#     gl.top_labels = False
#     gl.right_labels = False
#     gl.xlabel_style = {'size': 10}
#     gl.ylabel_style = {'size': 10}
    
#     # Legend (outside plot)
#     ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8, 
#               markerscale=5, ncol=1, title='Passes')
    
#     # Title
#     ax.set_title(f'All {len(all_passes)} SWOT Passes - Ground Tracks\n'
#                  f'Andaman Sea Region: {lat_bounds[0]}°-{lat_bounds[1]}°N, {lon_bounds[0]}°-{lon_bounds[1]}°E',
#                  fontsize=14, fontweight='bold')
    
#     plt.tight_layout()
#     plt.savefig('all_passes_andaman_sea.png', dpi=200, bbox_inches='tight')
#     plt.show()
    
#     # Print summary
#     print("\n" + "=" * 80)
#     print("SUMMARY: PASSES COVERING ANDAMAN SEA")
#     print("=" * 80)
    
#     if passes_in_region:
#         print(f"\n✓ {len(passes_in_region)} passes COVER the Andaman Sea:")
#         for pass_id, pts in sorted(passes_in_region, key=lambda x: -x[1]):
#             print(f"   Pass {pass_id:3d}: {pts:,} points in region")
    
#     if passes_outside_region:
#         print(f"\n✗ {len(passes_outside_region)} passes do NOT cover the region:")
#         print(f"   {passes_outside_region}")
    
#     print("=" * 80)
    
#     return passes_in_region, passes_outside_region


# # ====== RUN VISUALIZATION ======
# passes_in_region, passes_outside = visualize_all_passes(
#     unique_files, 
#     lat_bounds=(5.0, 20.0),   # Andaman Sea latitude
#     lon_bounds=(92.0, 100.0)  # Andaman Sea longitude
# )

In [10]:
# ===================================================================
# VISUALIZE ALL PASSES FROM SAME CYCLE ON ANDAMAN SEA (WITH ERROR HANDLING)
# ===================================================================

def visualize_single_cycle_all_passes(unique_files, cycle_id=None, 
                                       lat_bounds=(5.0, 20.0), lon_bounds=(92.0, 100.0)):
    """
    Visualize all passes from a single cycle on the Andaman Sea.
    Includes robust error handling for corrupted files.
    """
    
    FILL_THRESHOLD = 1e10
    
    # Get available cycles and their pass counts
    cycle_pass_counts = defaultdict(set)
    for f in unique_files:
        cycle_pass_counts[f['cycle']].add(f['pass_number'])
    
    # Sort cycles by number of passes
    cycles_sorted = sorted(cycle_pass_counts.items(), key=lambda x: -len(x[1]))
    
    print("=" * 80)
    print("AVAILABLE CYCLES (sorted by number of passes)")
    print("=" * 80)
    print(f"{'Cycle':<10} {'Passes Available':<20} {'Pass IDs'}")
    print("-" * 80)
    for cyc, passes in cycles_sorted[:15]:
        pass_list = sorted(passes)
        pass_str = str(pass_list) if len(pass_list) <= 8 else f"{pass_list[:4]}...{pass_list[-2:]}"
        print(f"{cyc:<10} {len(passes):<20} {pass_str}")
    print("=" * 80)
    
    # Select cycle
    if cycle_id is None:
        cycle_id = cycles_sorted[0][0]
        print(f"\nAuto-selected Cycle {cycle_id} (has most passes: {len(cycles_sorted[0][1])})")
    
    # Get files for selected cycle
    cycle_files = [f for f in unique_files if f['cycle'] == cycle_id]
    
    if len(cycle_files) == 0:
        print(f"\n⚠️ No files found for Cycle {cycle_id}")
        return None
    
    # Sort by pass number
    cycle_files.sort(key=lambda x: x['pass_number'])
    
    print(f"\n{'='*80}")
    print(f"CYCLE {cycle_id}: {len(cycle_files)} PASSES")
    print(f"{'='*80}")
    
    # ===================================================================
    # PRE-VALIDATE FILES - Try to open each and find alternatives if needed
    # ===================================================================
    validated_files = []
    
    for f in cycle_files:
        pass_id = f['pass_number']
        file_ok = False
        current_file = f
        
        # Try to open the file
        try:
            ds_test = xr.open_dataset(f['filepath'], group='left', engine='netcdf4')
            _ = ds_test['latitude'].values[0, 0]  # Try to read some data
            ds_test.close()
            file_ok = True
        except Exception as e:
            print(f"  ⚠️ Cycle {cycle_id} Pass {pass_id}: File corrupted, searching alternative...")
            
            # Find alternative files for this pass from other cycles
            alt_files = [af for af in unique_files 
                        if af['pass_number'] == pass_id and af['cycle'] != cycle_id]
            alt_files.sort(key=lambda x: abs(x['cycle'] - cycle_id))  # Sort by closest cycle
            
            for alt in alt_files:
                try:
                    ds_test = xr.open_dataset(alt['filepath'], group='left', engine='netcdf4')
                    _ = ds_test['latitude'].values[0, 0]
                    ds_test.close()
                    current_file = alt
                    file_ok = True
                    print(f"    ✓ Using Cycle {alt['cycle']} instead for Pass {pass_id}")
                    break
                except:
                    continue
            
            if not file_ok:
                print(f"    ✗ No valid file found for Pass {pass_id}")
        
        if file_ok:
            validated_files.append(current_file)
    
    if len(validated_files) == 0:
        print("\n⚠️ No valid files could be opened!")
        return None
    
    print(f"\nValidated {len(validated_files)}/{len(cycle_files)} files")
    
    # Create figure with subplots
    n_passes = len(validated_files)
    
    # Determine grid layout
    if n_passes <= 4:
        nrows, ncols = 1, max(n_passes, 1)
        figsize = (5 * ncols, 6)
    elif n_passes <= 8:
        nrows, ncols = 2, (n_passes + 1) // 2
        figsize = (5 * ncols, 10)
    elif n_passes <= 12:
        nrows, ncols = 3, (n_passes + 2) // 3
        figsize = (5 * ncols, 14)
    else:
        nrows, ncols = 4, (n_passes + 3) // 4
        figsize = (4 * ncols, 16)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize,
                              subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Flatten axes for easy iteration
    if n_passes == 1:
        axes = [axes]
    elif nrows == 1 or ncols == 1:
        axes = list(axes)
    else:
        axes = axes.flatten()
    
    # Track passes in region
    passes_in_region = []
    error_passes = []
    
    print(f"\nPlotting {n_passes} passes...")
    
    for idx, file_info in enumerate(validated_files):
        ax = axes[idx]
        pass_id = file_info['pass_number']
        used_cycle = file_info['cycle']
        
        try:
            # Load swath data
            ds_left = xr.open_dataset(file_info['filepath'], group='left', engine='netcdf4')
            ds_right = xr.open_dataset(file_info['filepath'], group='right', engine='netcdf4')
            
            # Get coordinates
            lat_left = np.where(np.abs(ds_left['latitude'].values) > FILL_THRESHOLD, 
                               np.nan, ds_left['latitude'].values)
            lon_left = np.where(np.abs(ds_left['longitude'].values) > FILL_THRESHOLD, 
                               np.nan, ds_left['longitude'].values)
            lat_right = np.where(np.abs(ds_right['latitude'].values) > FILL_THRESHOLD, 
                                np.nan, ds_right['latitude'].values)
            lon_right = np.where(np.abs(ds_right['longitude'].values) > FILL_THRESHOLD, 
                                np.nan, ds_right['longitude'].values)
            
            # Get SSHA
            ssha_left = np.where(np.abs(ds_left['ssha_karin'].values) > FILL_THRESHOLD,
                                np.nan, ds_left['ssha_karin'].values)
            ssha_right = np.where(np.abs(ds_right['ssha_karin'].values) > FILL_THRESHOLD,
                                 np.nan, ds_right['ssha_karin'].values)
            
            # Valid masks
            valid_left = np.isfinite(lat_left) & np.isfinite(lon_left) & np.isfinite(ssha_left)
            valid_right = np.isfinite(lat_right) & np.isfinite(lon_right) & np.isfinite(ssha_right)
            
            # Region masks
            region_left = valid_left & (lat_left >= lat_bounds[0]) & (lat_left <= lat_bounds[1]) & \
                         (lon_left >= lon_bounds[0]) & (lon_left <= lon_bounds[1])
            region_right = valid_right & (lat_right >= lat_bounds[0]) & (lat_right <= lat_bounds[1]) & \
                          (lon_right >= lon_bounds[0]) & (lon_right <= lon_bounds[1])
            
            pts_in_region = np.sum(region_left) + np.sum(region_right)
            
            # Setup map
            ax.set_extent([lon_bounds[0], lon_bounds[1], lat_bounds[0], lat_bounds[1]], 
                         crs=ccrs.PlateCarree())
            ax.add_feature(cfeature.LAND, facecolor='lightgray', edgecolor='black', linewidth=0.5)
            ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3)
            ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
            
            if pts_in_region > 0:
                passes_in_region.append((pass_id, pts_in_region))
                
                # Get data in region
                lat_plot = np.concatenate([lat_left[region_left], lat_right[region_right]])
                lon_plot = np.concatenate([lon_left[region_left], lon_right[region_right]])
                ssha_plot = np.concatenate([ssha_left[region_left], ssha_right[region_right]])
                
                # Color limits
                vmax = min(0.5, np.nanpercentile(np.abs(ssha_plot), 95))
                vmin = -vmax
                
                # Plot SSHA
                sc = ax.scatter(lon_plot, lat_plot, c=ssha_plot, 
                               cmap='RdBu_r', s=0.3, alpha=0.8,
                               vmin=vmin, vmax=vmax,
                               transform=ccrs.PlateCarree())
                
                cbar = plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.02)
                cbar.set_label('SSHA (m)', fontsize=8)
                cbar.ax.tick_params(labelsize=7)
                
                title_color = 'darkgreen'
                status = f"✓ {pts_in_region:,} pts"
            else:
                title_color = 'darkred'
                status = "✗ No data"
                ax.text(0.5, 0.5, 'No data\nin region', 
                       transform=ax.transAxes, ha='center', va='center',
                       fontsize=10, color='gray')
            
            # Gridlines
            gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5, linestyle='--')
            gl.top_labels = gl.right_labels = False
            gl.xlabel_style = {'size': 7}
            gl.ylabel_style = {'size': 7}
            
            # Title (show if using different cycle)
            cycle_note = f" (C{used_cycle})" if used_cycle != cycle_id else ""
            ax.set_title(f"Pass {pass_id}{cycle_note}\n{status}", 
                        fontsize=10, fontweight='bold', color=title_color)
            
            ds_left.close()
            ds_right.close()
            
        except Exception as e:
            error_passes.append(pass_id)
            ax.set_title(f"Pass {pass_id}\n⚠️ Error", fontsize=10, color='red')
            ax.text(0.5, 0.5, f'Error:\n{str(e)[:30]}', 
                   transform=ax.transAxes, ha='center', va='center',
                   fontsize=8, color='red')
            print(f"  ⚠️ Error Pass {pass_id}: {str(e)[:50]}")
    
    # Hide unused axes
    for idx in range(n_passes, len(axes)):
        axes[idx].set_visible(False)
    
    # Overall title
    fig.suptitle(f'SWOT Cycle {cycle_id} - All {n_passes} Passes\n'
                 f'Andaman Sea ({lat_bounds[0]}°-{lat_bounds[1]}°N, {lon_bounds[0]}°-{lon_bounds[1]}°E)',
                 fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.savefig(f'cycle_{cycle_id}_all_passes_andaman.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    # Summary
    print(f"\n{'='*80}")
    print(f"CYCLE {cycle_id} SUMMARY")
    print(f"{'='*80}")
    print(f"Total passes attempted: {len(cycle_files)}")
    print(f"Successfully plotted: {len(validated_files)}")
    print(f"Passes covering Andaman Sea: {len(passes_in_region)}")
    
    if passes_in_region:
        print(f"\n✓ Passes with data in region:")
        for pid, pts in sorted(passes_in_region, key=lambda x: -x[1]):
            print(f"   Pass {pid:3d}: {pts:,} points")
    
    if error_passes:
        print(f"\n⚠️ Passes with errors: {error_passes}")
    
    print(f"{'='*80}")
    
    return passes_in_region


# ====== RUN VISUALIZATION ======
passes_in_region = visualize_single_cycle_all_passes(
    unique_files,
    cycle_id=None,  # Auto-select, or specify like cycle_id=10
    lat_bounds=(5.0, 20.0),
    lon_bounds=(92.0, 100.0)
)

AVAILABLE CYCLES (sorted by number of passes)
Cycle      Passes Available     Pass IDs
--------------------------------------------------------------------------------


IndexError: list index out of range